# HUẤN LUYỆN & ĐÁNH GIÁ M4 (CPG-GNN) TRÊN BIGCLONEBENCH JAVA (GPU)

Notebook này thực hiện huấn luyện độc lập mô hình **M4 (CPG-GNN)** trên BigCloneBench Java.

### Thông tin cấu hình:
- **Mã mô hình:** M4
- **Tên phương pháp:** CPG-GNN
- **File cấu hình:** `configs/exp005_cpg_gnn_bcb.yaml`
- **Dataset liên kết:** `kichirou/semanticcodeclonedetection-dataset-m4`
- **Splits:** 30.000 Train, 3.000 Val, 5.000 Test cân bằng 50:50
- **Tự động lưu checkpoint & test metrics riêng biệt:** Kết quả lưu tại `/kaggle/working/`

---

In [ ]:
# 1. THIẾT LẬP MÔI TRƯỜNG & HÀM BỔ TRỢ STREAM LOG
import base64, io, json, os, shutil, subprocess, sys, tarfile, time, yaml
from pathlib import Path

def sh(cmd, **kw):
    print(f"\n$ {cmd}", flush=True)
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    log_file = Path("/kaggle/working/live_progress.log")
    with log_file.open("a", encoding="utf-8") as lf:
        for line in iter(proc.stdout.readline, ''):
            print(line, end='', flush=True)
            lf.write(line)
            lf.flush()
    proc.stdout.close()
    rc = proc.wait()
    if rc != 0:
        raise RuntimeError(f"Lệnh thất bại (rc={rc}): {cmd}")
    return ""

print("Python:", sys.version.split()[0], flush=True)

In [ ]:
# 2. GIẢI NÉN MÃ NGUỒN DỰ ÁN MỚI NHẤT (SELF-CONTAINED)
WORK_DIR = Path("/kaggle/working/project")
WORK_DIR.mkdir(parents=True, exist_ok=True)
CONFIG_FILE = "configs/exp005_cpg_gnn_bcb.yaml"
BUNDLE_B64 = "..."

print(f"Giải nén mã nguồn ({len(BUNDLE_B64)/1024:.1f} KB) vào {WORK_DIR}...")
buf = io.BytesIO(base64.b64decode(BUNDLE_B64))
with tarfile.open(fileobj=buf, mode="r:gz") as tar:
    tar.extractall(WORK_DIR)

os.chdir(WORK_DIR)
if str(WORK_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_DIR))
sh(f"ls -la {CONFIG_FILE}")

In [ ]:
# 3. CÀI ĐẶT THƯ VIỆN & XÁC THỰC GPU
sh("pip install -q 'tree-sitter>=0.23.0' 'tree-sitter-java>=0.23.5' 'tree-sitter-cpp>=0.23.0' 'datasets>=2.14.0' 'transformers>=4.30.0' 'torch_geometric' 'pyarrow' 'tqdm' 'pyyaml' 'scikit-learn'")

import torch
assert torch.cuda.is_available(), "Cần bật GPU trong Settings -> Accelerator -> GPU (T4/P100)!"
gpu_name = torch.cuda.get_device_name(0)
vram_gb = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2)
print(f"GPU SẴN SÀNG: {gpu_name} | VRAM: {vram_gb} GB")
sh("nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv")

In [ ]:
# 4. LIÊN KẾT ĐẶC TRƯNG TỪ DATASET NẾU CÓ
import numpy as np
ds_candidates = [
    Path("/kaggle/input/datasets/kichirou/semanticcodeclonedetection-dataset-m4"),
    Path("/kaggle/input/semanticcodeclonedetection-dataset-m4"),
]
ds_root = None
for d in ds_candidates:
    if d.exists():
        ds_root = d
        break

print(f"Thư mục dataset phát hiện: {ds_root}")
if ds_root:
    sh(f"ls -la {ds_root}/")
# Liên kết CPG Graphs cho M4
os.makedirs("artifacts", exist_ok=True)
cpg_src = ds_root / "bcb_cpg_export" if ds_root else None
if cpg_src and cpg_src.exists():
    print(f"Phát hiện thư mục CPG export tại: {cpg_src}")
    inner = cpg_src / "cpg_export"
    dest = Path("artifacts/cpg_export")
    if inner.exists():
        sh(f"cp -r {inner} artifacts/")
    else:
        sh(f"cp -r {cpg_src} artifacts/cpg_export")
    cpg_count = len(list(dest.glob("*.json")))
    print(f"Đã liên kết {cpg_count:,} đồ thị CPG JSON vào {dest}!")
elif (ds_root / "bcb_cpg_export.tar.gz").exists() if ds_root else False:
    sh(f"tar -xzf {ds_root}/bcb_cpg_export.tar.gz -C artifacts/")
    print("Đã giải nén bcb_cpg_export.tar.gz vào artifacts/!")
else:
    print("Cảnh báo: Không tìm thấy tài nguyên CPG export!")


In [ ]:
# 5. TẢI CÁC SPLIT DỮ LIỆU ĐỂ HUẤN LUYỆN (TRAIN 30.000, VAL 3.000, TEST 5.000 CÂN BẰNG 50:50)
data_raw = Path("data/raw/bigclonebench")
data_raw.mkdir(parents=True, exist_ok=True)
print("Tải các phân tập dữ liệu BigCloneBench (Train: 30k, Val: 3k, Test: 5k) cân bằng 50:50...")
sh("python scripts/download_bcb_subsets.py --splits-custom --train-samples 30000 --val-samples 3000 --test-samples 5000")
sh("ls -lh data/raw/bigclonebench/")
assert (data_raw / 'bcb_train.parquet').exists() and (data_raw / 'bcb_train.parquet').stat().st_size > 0, 'bcb_train.parquet phai ton tai va co du lieu!'

In [ ]:
# 6. PREFLIGHT & DIAGNOSTIC CHECK
print("--- PREFLIGHT M4 ---")
sh(f"python -m src.experiments.bcb_runner --config {CONFIG_FILE} --mode preflight --device cuda")
print("\n--- DIAGNOSTIC CHECK M4 ---")
sh(f"python -m src.experiments.bcb_runner --config {CONFIG_FILE} --mode diagnostic --device cuda")

In [ ]:
# 7. TIẾN HÀNH HUẤN LUYỆN M4 (CPG-GNN) TRÊN GPU
print("=" * 60)
print("BẮT ĐẦU HUẤN LUYỆN M4 (CPG-GNN)...")
print("=" * 60)
TIME_BUDGET_S = 9 * 3600
try:
    sh(f"python -m src.experiments.bcb_runner --config {CONFIG_FILE} --mode train --device cuda --time-budget-s {TIME_BUDGET_S}")
finally:
    with open(CONFIG_FILE, 'r', encoding='utf-8') as f:
        exp_id = yaml.safe_load(f)['experiment']['id'].lower().replace('-', '_')
    res_dir = Path('results') / exp_id
    dest_dir = Path('/kaggle/working/m4_results')
    dest_dir.mkdir(parents=True, exist_ok=True)
    if res_dir.exists():
        for f in res_dir.glob('*'):
            sh(f'cp -r {f} {dest_dir}/')
            if f.suffix in ('.json', '.pt'):
                sh(f'cp {f} /kaggle/working/')
    sh(f'tar -czf /kaggle/working/m4_results.tar.gz results/')
    print(f'>>> ĐÃ SAO LƯU CHECKPOINT VÀ KẾT QUẢ HUẤN LUYỆN CỦA M4 VÀO /kaggle/working/')

In [ ]:
# 8. ĐÁNH GIÁ MÔ HÌNH M4 TRÊN TẬP TEST
print("=" * 60)
print("ĐÁNH GIÁ MÔ HÌNH M4 TRÊN TẬP TEST...")
print("=" * 60)
sh(f"python -m src.experiments.bcb_runner --config {CONFIG_FILE} --mode evaluate --device cuda")

In [ ]:
# 9. ĐÓNG GÓI & TỔNG KẾT KẾT QUẢ VÀO /kaggle/working/
with open(CONFIG_FILE, 'r', encoding='utf-8') as f:
    exp_id = yaml.safe_load(f)['experiment']['id'].lower().replace('-', '_')
res_dir = Path('results') / exp_id
dest_dir = Path('/kaggle/working/m4_results')
dest_dir.mkdir(parents=True, exist_ok=True)

if res_dir.exists():
    for f in res_dir.glob('*'):
        sh(f'cp -r {f} {dest_dir}/')
        if f.suffix in ('.json', '.pt'):
            sh(f'cp {f} /kaggle/working/')

TAR_RESULTS = Path('/kaggle/working/m4_results.tar.gz')
sh(f'tar -czf {TAR_RESULTS} results/')
sh('ls -lh /kaggle/working/*.json /kaggle/working/*.tar.gz 2>/dev/null || ls -lh /kaggle/working/')

if (res_dir / 'test_metrics.json').exists():
    metrics = json.loads((res_dir / 'test_metrics.json').read_text(encoding='utf-8'))
    print('\nBẢNG TEST METRICS CỦA M4:\n', json.dumps(metrics, indent=2))
print('\nHUẤN LUYỆN VÀ ĐÁNH GIÁ M4 HOÀN TẤT THÀNH CÔNG!')